<a href="https://colab.research.google.com/github/mizomizo1/FEVR_analysis/blob/test_branch/revise_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training and Validation

In [ ]:
# @title
!hostname
!python -V
!pwd

import csv
import os
import random
import sys
from datetime import datetime, timezone
import platform
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from google.colab import drive
from PIL import Image
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from tqdm.notebook import tqdm

torch.manual_seed(123)  # 乱数のシードを設定

3f10ea657058
Python 3.12.12
/content


In [ ]:
drive.mount('/content/drive')
project_root = "/content/drive/MyDrive/ScientificReport/re-analysis/version2/"
train_val_root = os.path.join(project_root, "train_val")
code_root = os.path.join(project_root, "code")
data_root = os.path.join(train_val_root, "data")
data_path = os.path.join(data_root, "fig")
split_csv_path = os.path.join(data_root, "meta", "fixed", "fixed_family_level_split.csv")

# 査読対応で追跡しやすいように、出力は review_outputs 以下へ集約する
experiment_name = "family_level_split_efficientnetb3_seed123"
review_output_root = os.path.join(train_val_root, "review_outputs", experiment_name)
model_path = os.path.join(review_output_root, "models")
log_path = os.path.join(review_output_root, "logs")
wandb_path = os.path.join(log_path, "wandb")
metadata_output_path = os.path.join(review_output_root, "metadata_snapshots")
prediction_root = os.path.join(review_output_root, "predictions")
prediction_csv_path = os.path.join(prediction_root, "tables")
gradcam_root = os.path.join(review_output_root, "gradcam")
figure_output_path = os.path.join(review_output_root, "figures")
training_results_path = os.path.join(log_path, "training_results.csv")
review_summary_path = os.path.join(log_path, "reproducibility_summary.md")

for path in [
    review_output_root,
    model_path,
    log_path,
    wandb_path,
    metadata_output_path,
    prediction_root,
    prediction_csv_path,
    gradcam_root,
    figure_output_path,
]:
    os.makedirs(path, exist_ok=True)

if os.path.exists(project_root):
    os.chdir(project_root)

print(f"project_root: {project_root}")
print(f"train_val_root: {train_val_root}")
print(f"data_path: {data_path}")
print(f"split_csv_path: {split_csv_path}")
print(f"review_output_root: {review_output_root}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
project_root: /content/drive/MyDrive/ScientificReport/re-analysis/version2/
train_val_root: /content/drive/MyDrive/ScientificReport/re-analysis/version2/train_val
data_path: /content/drive/MyDrive/ScientificReport/re-analysis/version2/train_val/data/fig
split_csv_path: /content/drive/MyDrive/ScientificReport/re-analysis/version2/train_val/data/meta/fixed/fixed_family_level_split.csv
review_output_root: /content/drive/MyDrive/ScientificReport/re-analysis/version2/train_val/review_outputs/family_level_split_efficientnetb3_seed123


In [ ]:
REVIEW_CONFIG = {
    "seed": 123,
    "architecture": "EfficientNet B3",
    "pretrained_weights": "IMAGENET1K_V1",
    "image_size": (300, 300),
    "filename_column": "filename",
    "label_column": "label",
    "split_column": "fixed_family_level_split",
    "train_split_name": "train",
    "validation_split_name": "validation",
    "class_to_idx": {"F": 0, "N": 1},
    "batch_size": 8,
    "epochs": 100,
    "learning_rate": 0.001,
    "optimizer": "Adam",
    "loss_function": "CrossEntropyLoss"
}

def seed_worker(worker_id):
    worker_seed = REVIEW_CONFIG["seed"] + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)

def set_reproducibility(seed=REVIEW_CONFIG["seed"]):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as exc:
        print(f"deterministic_algorithmsは一部未対応です: {exc}")
    return seed

class MetadataImageDataset(Dataset):
    def __init__(self, rows, image_dir, transform=None, class_to_idx=None):
        self.image_dir = image_dir
        self.transform = transform
        self.class_to_idx = class_to_idx or {"F": 0, "N": 1}
        self.classes = [label for label, _ in sorted(self.class_to_idx.items(), key=lambda item: item[1])]
        self.samples = []
        self.targets = []
        self.records = []
        missing_files = []

        for row in rows:
            filename = row[REVIEW_CONFIG["filename_column"]]
            label = row[REVIEW_CONFIG["label_column"]]
            image_path = os.path.join(image_dir, filename)
            if not os.path.exists(image_path):
                missing_files.append(filename)
                continue
            target = self.class_to_idx[label]
            self.samples.append((image_path, target))
            self.targets.append(target)
            self.records.append(row)

        if missing_files:
            raise FileNotFoundError(f"metadataにあるのに画像が見つからないファイルがあります: {missing_files[:5]}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, target = self.samples[idx]
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, target

set_reproducibility()
print(f"review_seed:{REVIEW_CONFIG['seed']}")
print(f"deterministic_algorithms:{torch.are_deterministic_algorithms_enabled()}")
print(f"cudnn.deterministic:{torch.backends.cudnn.deterministic}")
print(f"cudnn.benchmark:{torch.backends.cudnn.benchmark}")

# 事前学習済みモデルの読み込み (EfficientNet B3を使用)

model = models.efficientnet_b3(weights='IMAGENET1K_V1')  # 'pretrained'を'weights'に変更

# 最終層の変更 (ここでは2クラス分類に変更)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 2)

# データセットの読み込みと前処理
# 訓練データ用の拡張
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

train_transform = transforms.Compose([
    transforms.Resize(size=(300,300)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),# 検証データ用の拡張 (augmentationは行わない)
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 検証データ用の拡張 (augmentationは行わない)
val_transform = transforms.Compose([
    transforms.Resize(size=(300,300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# データセットのパス
data_dir = data_path

# family-levelで固定したsplit CSVを読み込む
with open(split_csv_path, newline="") as f:
    metadata_rows = list(csv.DictReader(f))

split_column = REVIEW_CONFIG["split_column"]
label_column = REVIEW_CONFIG["label_column"]
train_split_name = REVIEW_CONFIG["train_split_name"]
validation_split_name = REVIEW_CONFIG["validation_split_name"]

train_rows = [row for row in metadata_rows if row[split_column] == train_split_name]
val_rows = [row for row in metadata_rows if row[split_column] == validation_split_name]

train_label_counts = {}
for row in train_rows:
    label = row[label_column]
    train_label_counts[label] = train_label_counts.get(label, 0) + 1

val_label_counts = {}
for row in val_rows:
    label = row[label_column]
    val_label_counts[label] = val_label_counts.get(label, 0) + 1

train_base_dataset = MetadataImageDataset(train_rows, data_dir, transform=train_transform, class_to_idx=REVIEW_CONFIG["class_to_idx"])
val_base_dataset = MetadataImageDataset(val_rows, data_dir, transform=val_transform, class_to_idx=REVIEW_CONFIG["class_to_idx"])

# 既存セルとの互換性のためSubsetで包む
train_dataset = Subset(train_base_dataset, list(range(len(train_base_dataset))))
val_dataset = Subset(val_base_dataset, list(range(len(val_base_dataset))))

# DataLoaderを作成
loader_generator = torch.Generator().manual_seed(REVIEW_CONFIG["seed"])
train_loader = DataLoader(train_dataset, batch_size=REVIEW_CONFIG["batch_size"], shuffle=True, num_workers=os.cpu_count(), pin_memory=True, worker_init_fn=seed_worker, generator=loader_generator)
val_loader = DataLoader(val_dataset, batch_size=REVIEW_CONFIG["batch_size"], shuffle=False, num_workers=os.cpu_count(), pin_memory=True, worker_init_fn=seed_worker)
val_1_loader = DataLoader(train_dataset, batch_size=REVIEW_CONFIG["batch_size"], shuffle=False, worker_init_fn=seed_worker)

print(f"DataLoaderを作成しました。")
print(f"split_csv:{split_csv_path}")
print(f"split_column:{split_column}")
print(f"random_seed:{REVIEW_CONFIG['seed']}")
print(f"train_datasetの写真枚数は{len(train_dataset)}枚")
print(f"val_datasetの写真枚数は{len(val_dataset)}枚")
print(f"train label counts:{train_label_counts}")
print(f"validation label counts:{val_label_counts}")

# 査読でそのまま貼れるように、実行環境と再現性情報をMarkdown形式で出力する

review_environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "scikit_learn": sklearn.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "device": str(device),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "cpu_count": os.cpu_count(),
}

review_lines = [
    "### Reproducibility Summary",
    f"- Executed at (UTC): {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}",
    f"- Python: {review_environment['python']}",
    f"- Platform: {review_environment['platform']}",
    f"- PyTorch: {review_environment['torch']}",
    f"- torchvision: {review_environment['torchvision']}",
    f"- scikit-learn: {review_environment['scikit_learn']}",
    f"- CUDA available: {review_environment['cuda_available']}",
    f"- CUDA version: {review_environment['cuda_version']}",
    f"- cuDNN version: {review_environment['cudnn_version']}",
    f"- Device used: {review_environment['device']} ({review_environment['gpu_name']})",
    f"- CPU cores: {review_environment['cpu_count']}",
    f"- Random seed: {REVIEW_CONFIG['seed']}",
    f"- Deterministic algorithms: {torch.are_deterministic_algorithms_enabled()}",
    f"- cuDNN deterministic: {torch.backends.cudnn.deterministic}",
    f"- cuDNN benchmark: {torch.backends.cudnn.benchmark}",
    f"- Architecture: {REVIEW_CONFIG['architecture']}",
    f"- Pretrained weights: {REVIEW_CONFIG['pretrained_weights']}",
    f"- Image size: {REVIEW_CONFIG['image_size']}",
    f"- Data path: {data_path}",
    f"- Split CSV path: {split_csv_path}",
    f"- Split column: {REVIEW_CONFIG['split_column']}",
    f"- Model output path: {model_path}",
    f"- Log path: {log_path}",
    f"- Metadata output path: {metadata_output_path}",
    f"- Prediction CSV path: {prediction_csv_path}",
    f"- Grad-CAM output path: {gradcam_root}",
    f"- Figure output path: {figure_output_path}",
    f"- Dataset split: train {len(train_dataset)} / validation {len(val_dataset)}",
    f"- Train label counts: {train_label_counts}",
    f"- Validation label counts: {val_label_counts}",
    f"- Batch size: {REVIEW_CONFIG['batch_size']}",
    f"- Epochs: {REVIEW_CONFIG['epochs']}",
    f"- Learning rate: {REVIEW_CONFIG['learning_rate']}",
    f"- Optimizer: {REVIEW_CONFIG['optimizer']}",
    f"- Loss function: {REVIEW_CONFIG['loss_function']}",
    f"- Train transform: {train_transform}",
    f"- Validation transform: {val_transform}",
]

review_text = "\n".join(review_lines)
print(review_text)
with open(review_summary_path, "w", encoding="utf-8") as f:
    f.write(review_text)
print(f"Saved reproducibility summary to: {review_summary_path}")
review_text

# 損失関数と最適化手法の設定
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=REVIEW_CONFIG["learning_rate"])
print(f"損失関数と最適化手法の設定が完了しました。")
print(f"損失関数:{REVIEW_CONFIG['loss_function']}")
print(f"最適化手法:{REVIEW_CONFIG['optimizer']}")
print(f"学習率:{REVIEW_CONFIG['learning_rate']}")
!pip install wandb
import wandb

# WandBの初期化
wandb.login()
wandb.init(project="FEVR-classification_revise",
            name=experiment_name,
            dir=wandb_path,
            config={
                "learning_rate": REVIEW_CONFIG["learning_rate"],
                "architecture": REVIEW_CONFIG["architecture"],
                "dataset": f"fig + {REVIEW_CONFIG['split_column']}",
                "epochs": REVIEW_CONFIG["epochs"],
                "batch_size": REVIEW_CONFIG["batch_size"],
                "optimizer": REVIEW_CONFIG["optimizer"],
                "loss_function": REVIEW_CONFIG["loss_function"]
            })
#torch.random.initial_seed()
# 学習のループ

num_epochs = REVIEW_CONFIG["epochs"]  # エポック数

train_losses = []
train_accs = []
train_aucs = []
val_losses = []
val_accs = []
val_aucs = []

for epoch in range(num_epochs):
    torch.manual_seed(epoch + REVIEW_CONFIG["seed"])
    # 訓練フェーズ
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []

    for images, labels in tqdm(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(torch.softmax(outputs, dim=1)[:, 1].detach().cpu().numpy())  # 確率値を取得

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    try:
        train_auc = roc_auc_score(all_labels, all_preds)
    except ValueError:
        train_auc = 0.0  # AUCが計算できない場合は0とする

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    train_aucs.append(train_auc)

    # 検証フェーズ
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(torch.softmax(outputs, dim=1)[:, 1].detach().cpu().numpy())

    val_loss = running_loss / len(val_loader)
    val_acc = 100 * correct / total
    try:
        val_auc = roc_auc_score(all_labels, all_preds)
    except ValueError:
        val_auc = 0.0

    val_losses.append(val_loss)
    val_accs.append(val_acc)
    val_aucs.append(val_auc)
    wandb.log({
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_auc": train_auc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_auc": val_auc
    })
    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, Train AUC: {train_auc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%, Val AUC: {val_auc:.4f}')
    model_save_path = os.path.join(model_path, f'model_{epoch+1}.pth')
    torch.save(model.state_dict(), model_save_path)
    print("save the model")


print('Finished Training')

review_seed:123
deterministic_algorithms:True
cudnn.deterministic:True
cudnn.benchmark:False
DataLoaderを作成しました。
split_csv:/content/drive/MyDrive/ScientificReport/re-analysis/version2/train_val/data/meta/fixed/fixed_family_level_split.csv
split_column:fixed_family_level_split
random_seed:123
train_datasetの写真枚数は125枚
val_datasetの写真枚数は31枚
train label counts:{'F': 51, 'N': 74}
validation label counts:{'F': 13, 'N': 18}
### Reproducibility Summary
- Executed at (UTC): 2026-03-21 14:07:56
- Python: 3.12.12
- Platform: Linux-6.6.113+-x86_64-with-glibc2.35
- PyTorch: 2.10.0+cu128
- torchvision: 0.25.0+cu128
- scikit-learn: 1.6.1
- CUDA available: True
- CUDA version: 12.8
- cuDNN version: 91002
- Device used: cuda (Tesla T4)
- CPU cores: 8
- Random seed: 123
- Deterministic algorithms: True
- cuDNN deterministic: True
- cuDNN benchmark: False
- Architecture: EfficientNet B3
- Pretrained weights: IMAGENET1K_V1
- Image size: (300, 300)
- Data path: /content/drive/MyDrive/ScientificReport/re-analy

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: h6i1d7e (h6i1d7e-kansai-medical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  0%|          | 0/16 [00:00<?, ?it/s]

Epoch [1/100], Train Loss: 0.6273, Train Acc: 68.00%, Train AUC: 0.6948, Val Loss: 0.4146, Val Acc: 77.42%, Val AUC: 0.9060
save the model


  0%|          | 0/16 [00:00<?, ?it/s]

...

Epoch [98/100], Train Loss: 0.0443, Train Acc: 98.40%, Train AUC: 0.9984, Val Loss: 0.5072, Val Acc: 83.87%, Val AUC: 0.9017
save the model


  0%|          | 0/16 [00:00<?, ?it/s]

Epoch [99/100], Train Loss: 0.0361, Train Acc: 99.20%, Train AUC: 0.9997, Val Loss: 0.5925, Val Acc: 83.87%, Val AUC: 0.9573
save the model


  0%|          | 0/16 [00:00<?, ?it/s]

Epoch [100/100], Train Loss: 0.0186, Train Acc: 99.20%, Train AUC: 1.0000, Val Loss: 0.7335, Val Acc: 80.65%, Val AUC: 0.9957
save the model
Finished Training


## Grad-CAM(meta)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torchvision import transforms, models

from sklearn.metrics import roc_auc_score, confusion_matrix, precision_score, f1_score, precision_recall_curve, auc as pr_auc
# モデルの読み込み
best_model_path = os.path.join("train_val", "review_outputs", experiment_name, "models", "model_24.pth")  # best_model_fileに置き換えてください
model = models.efficientnet_b3(weights='IMAGENET1K_V1')
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 2)
model.load_state_dict(torch.load(best_model_path))
model = model.to(device)
model.eval()  # 評価モードに設定
class CustomDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.filenames = os.listdir(data_dir)

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_name = os.path.join(self.data_dir, self.filenames[idx])
        image = Image.open(img_name).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, self.filenames[idx]
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_handles = []

        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        forward_handle = self.target_layer.register_forward_hook(forward_hook)
        backward_handle = self.target_layer.register_backward_hook(backward_hook)

        self.hook_handles.append(forward_handle)
        self.hook_handles.append(backward_handle)

    def generate(self, input_image, target_class=None):
        # 順伝播
        output = self.model(input_image)

        if target_class is None:
            target_class = output.argmax(dim=1).item()

        # 逆伝播
        self.model.zero_grad()
        target = output[0][target_class]
        target.backward()

        # 勾配と活性化を取得
        gradients = self.gradients[0]
        activations = self.activations[0]

        # Grad-CAM
        weights = torch.mean(gradients, dim=[1, 2], keepdim=True)
        grad_cam = torch.sum(weights * activations, dim=0)
        grad_cam = F.relu(grad_cam)
        grad_cam = F.interpolate(grad_cam.unsqueeze(0).unsqueeze(0),
                                 size=input_image.shape[2:], mode="bilinear", align_corners=False)
        grad_cam = grad_cam.squeeze().cpu().numpy()

        # 正規化
        grad_cam = (grad_cam - grad_cam.min()) / (grad_cam.max() - grad_cam.min())
        return grad_cam

    def remove_hooks(self):
        for handle in self.hook_handles:
            handle.remove()

def process_directory_with_grad_cam_using_metadata(
    model,
    directory_path,
    metadata_csv_path,
    split_name,           # "train" or "validation"
    class_names,
    predict_path,
    csv_path
):
    # metadata読み込み
    meta_df = pd.read_csv(metadata_csv_path)

    # 必要なsplitだけ使用
    meta_df = meta_df[meta_df["fixed_family_level_split"] == split_name].copy()

    # label を数値化: F=0, N=1 (Consistent with REVIEW_CONFIG for positive class 'N')
    meta_df["true_label_num"] = meta_df["label"].map({"F": 0, "N": 1})

    # filename -> true label の辞書
    label_map = dict(zip(meta_df["filename"], meta_df["true_label_num"]))
    true_label_name_map = dict(zip(meta_df["filename"], meta_df["label"]))

    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((300, 300)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    os.makedirs(predict_path, exist_ok=True)
    results = []

    target_filenames = meta_df["filename"].tolist()

    for filename in tqdm(target_filenames):
        file_path = os.path.join(directory_path, filename)

        if not os.path.exists(file_path):
            print(f"skip: not found -> {file_path}")
            continue

        image = cv2.imread(file_path)
        if image is None:
            print(f"skip: failed to read -> {file_path}")
            continue

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_tensor = transform(image_rgb).unsqueeze(0).to(device)

        target_layer = model.features[-1]
        grad_cam = GradCAM(model, target_layer)
        cam = grad_cam.generate(input_image=image_tensor, target_class=1) # Specify N (index 1) as target class for grad-cam

        with torch.no_grad():
            outputs = model(image_tensor)
            probabilities = F.softmax(outputs, dim=1)
            predicted_class_idx = torch.argmax(probabilities).item()

            # Probability of 'N' (positive class, index 1) for metrics
            prob_n = probabilities[0, 1].item()
            pred_label = class_names[predicted_class_idx]

        image_rgb_resized = cv2.resize(image_rgb, (300, 300))
        plt.figure(figsize=(12, 4))

        plt.subplot(131)
        plt.imshow(image_rgb_resized)
        plt.title(filename)
        plt.axis("off")

        plt.subplot(132)
        plt.imshow(cam, cmap="jet")
        plt.title("Grad-CAM")
        plt.axis("off")

        plt.subplot(133)
        plt.imshow(image_rgb_resized)
        plt.imshow(cam, cmap="jet", alpha=0.4)
        plt.title(
            f"True: {true_label_name_map[filename]}\n"
            f"Pred: {pred_label}\n"
            f"N_prob={prob_n*100:.2f}%" # Changed to N_prob
        )
        plt.axis("off")

        save_path = os.path.join(predict_path, f"{pred_label}_{filename}.png")
        plt.savefig(save_path)
        plt.close()
        grad_cam.remove_hooks()

        results.append({
            "filename": filename,
            "split": split_name,
            "true_label": true_label_name_map[filename],
            "true_label_num": label_map[filename],
            "predicted_class": pred_label,
            "predicted_class_num": 1 if pred_label == "N" else 0, # N is positive class
            "probability_N": prob_n, # Changed to probability_N
        })

    df = pd.DataFrame(results)
    df.to_csv(csv_path, index=False)
    print(f"✅ 予測結果を保存: {csv_path}")

    if len(df) > 0 and df["true_label_num"].nunique() == 2:
        y_true = df["true_label_num"]
        y_pred = df["predicted_class_num"]
        y_score = df["probability_N"] # Probability of class N, which is 1 (positive class)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        # Calculate metrics
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
        f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
        roc_auc = roc_auc_score(y_true, y_score)

        # PR-AUC
        precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_score, pos_label=1)
        pr_auc_score = pr_auc(recall_curve, precision_curve)

        # Bootstrapping for 95% CI of ROC AUC
        n_bootstraps = 1000
        bootstrapped_scores = []

        for i in range(n_bootstraps):
            indices = np.random.choice(len(y_true), len(y_true), replace=True)
            y_true_resample = y_true.iloc[indices]
            y_score_resample = y_score.iloc[indices]

            if y_true_resample.nunique() > 1:
                bootstrapped_scores.append(roc_auc_score(y_true_resample, y_score_resample))
            else:
                pass # Skip if only one class in resample

        if bootstrapped_scores:
            sorted_scores = np.array(bootstrapped_scores)
            sorted_scores.sort()
            ci_lower = sorted_scores[int(0.025 * len(sorted_scores))]
            ci_upper = sorted_scores[int(0.975 * len(sorted_scores))]
        else:
            ci_lower = np.nan
            ci_upper = np.nan


        print(f"📊 Sensitivity: {sensitivity:.3f}")
        print(f"📊 Specificity: {specificity:.3f}")
        print(f"📊 Precision: {precision:.3f}")
        print(f"📊 F1-score: {f1:.3f}")
        print(f"📈 ROC AUC: {roc_auc:.3f} (95% CI: {ci_lower:.3f}-{ci_upper:.3f})")
        print(f"📈 PR-AUC: {pr_auc_score:.3f}")

        metrics = pd.DataFrame([{
            "split": split_name,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Precision": precision,
            "F1-score": f1,
            "AUC": roc_auc,
            "AUC_CI_Lower": ci_lower,
            "AUC_CI_Upper": ci_upper,
            "PR_AUC": pr_auc_score,
        }])
        metrics_path = csv_path.replace(".csv", "_metrics.csv")
        metrics.to_csv(metrics_path, index=False)
        print(f"✅ 評価指標を保存: {metrics_path}")
    else:
        print("⚠️ AUC/confusion matrix/other metrics はクラスが片側しかないため計算しませんでした。")



# validationで実行
predict_path = os.path.join(gradcam_root, "validation")
csv_path = os.path.join(prediction_csv_path, "gradcam_results_validation.csv") # Changed csv_path for validation
class_names = ["F", "N"]

process_directory_with_grad_cam_using_metadata(
    model=model,
    directory_path=os.path.join("train_val", "data", "fig"),
    metadata_csv_path=os.path.join("train_val", "data", "meta", "fixed", "fixed_family_level_split.csv"),
    split_name="validation",
    class_names=class_names,
    predict_path=predict_path,
    csv_path=csv_path
)


# trainingで実行
predict_path = os.path.join(gradcam_root, "train")
csv_path = os.path.join(prediction_csv_path, "gradcam_results_training.csv")
class_names = ["F", "N"]

process_directory_with_grad_cam_using_metadata(
    model=model,
    directory_path=os.path.join("train_val", "data", "fig"),
    metadata_csv_path=os.path.join("train_val", "data", "meta", "fixed", "fixed_family_level_split.csv"),
    split_name="train",
    class_names=class_names,
    predict_path=predict_path,
    csv_path=csv_path
)

  0%|          | 0/31 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1867: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)
/tmp/ipykernel_21530/3227808856.py:89: RuntimeWarning: invalid value encountered in divide
  grad_cam = (grad_cam - grad_cam.min()) / (grad_cam.max() - grad_cam.min())
100%|██████████| 31/31 [00:15<00:00,  1.98it/s]


✅ 予測結果を保存: /content/drive/MyDrive/ScientificReport/re-analysis/version2/train_val/review_outputs/family_level_split_efficientnetb3_seed123/predictions/tables/gradcam_results_validation.csv
📊 Sensitivity: 0.944
📊 Specificity: 0.692
📊 Precision: 0.810
📊 F1-score: 0.872
📈 ROC AUC: 0.842 (95% CI: 0.620-1.000)
📈 PR-AUC: 0.765
✅ 評価指標を保存: /content/drive/MyDrive/ScientificReport/re-analysis/version2/train_val/review_outputs/family_level_split_efficientnetb3_seed123/predictions/tables/gradcam_results_validation_metrics.csv


  0%|          | 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1867: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)
/tmp/ipykernel_21530/3227808856.py:89: RuntimeWarning: invalid value encountered in divide
  grad_cam = (grad_cam - grad_cam.min()) / (grad_cam.max() - grad_cam.min())
100%|██████████| 125/125 [01:02<00:00,  2.01it/s]


✅ 予測結果を保存: /content/drive/MyDrive/ScientificReport/re-analysis/version2/train_val/review_outputs/family_level_split_efficientnetb3_seed123/predictions/tables/gradcam_results_training.csv
📊 Sensitivity: 1.000
📊 Specificity: 1.000
📊 Precision: 1.000
📊 F1-score: 1.000
📈 ROC AUC: 1.000 (95% CI: 1.000-1.000)
📈 PR-AUC: 1.000
✅ 評価指標を保存: /content/drive/MyDrive/ScientificReport/re-analysis/version2/train_val/review_outputs/family_level_split_efficientnetb3_seed123/predictions/tables/gradcam_results_training_metrics.csv


# Test

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torchvision import transforms, models

from sklearn.metrics import roc_auc_score, confusion_matrix, precision_score, f1_score, precision_recall_curve, auc as pr_auc

In [ ]:
drive.mount('/content/drive')
project_root = "/content/drive/MyDrive/ScientificReport/re-analysis/version2/"
test_root = os.path.join(project_root, "test")
code_root = os.path.join(project_root, "code")
data_root = os.path.join(test_root, "data")
data_CAMpath = os.path.join(data_root, "fig")
split_csv_CAMpath = os.path.join(data_root, "meta", "fixed", "fixed_family_level_split.csv")

# 査読対応で追跡しやすいように、出力は review_outputs 以下へ集約する
experiment_name = "family_level_split_efficientnetb3_seed123"
review_output_root = os.path.join(test_root, "review_outputs", experiment_name)
prediction_root = os.path.join(review_output_root, "predictions")
prediction_csv_path = os.path.join(prediction_root, "tables")
gradcam_root = os.path.join(review_output_root, "gradcam")
review_summary_path = os.path.join(log_path, "reproducibility_summary.md")
for path in [
    review_output_root,
    prediction_root,
    prediction_csv_path,
    gradcam_root,
]:
    os.makedirs(path, exist_ok=True)

if os.path.exists(project_root):
    os.chdir(project_root)

print(f"project_root: {project_root}")
print(f"train_val_root: {test_root}")
print(f"data_path: {data_CAMpath}")
print(f"split_csv_path: {split_csv_CAMpath}")
print(f"review_output_root: {review_output_root}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
project_root: /content/drive/MyDrive/ScientificReport/re-analysis/version2/
train_val_root: /content/drive/MyDrive/ScientificReport/re-analysis/version2/test
data_path: /content/drive/MyDrive/ScientificReport/re-analysis/version2/test/data/fig
split_csv_path: /content/drive/MyDrive/ScientificReport/re-analysis/version2/test/data/meta/fixed/fixed_family_level_split.csv
review_output_root: /content/drive/MyDrive/ScientificReport/re-analysis/version2/test/review_outputs/family_level_split_efficientnetb3_seed123


In [ ]:
REVIEW_CONFIG = {
    "seed": 123,
    "architecture": "EfficientNet B3",
    "pretrained_weights": "IMAGENET1K_V1",
    "image_size": (300, 300),
    "filename_column": "filename",
    "label_column": "label",
    "split_column": "fixed_family_level_split",
    "train_split_name": "train",
    "validation_split_name": "validation",
    "class_to_idx": {"F": 0, "N": 1},
    "batch_size": 8,
    "epochs": 100,
    "learning_rate": 0.001,
    "optimizer": "Adam",
    "loss_function": "CrossEntropyLoss"
}

def seed_worker(worker_id):
    worker_seed = REVIEW_CONFIG["seed"] + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)

def set_reproducibility(seed=REVIEW_CONFIG["seed"]):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as exc:
        print(f"deterministic_algorithmsは一部未対応です: {exc}")
    return seed

class MetadataImageDataset(Dataset):
    def __init__(self, rows, image_dir, transform=None, class_to_idx=None):
        self.image_dir = image_dir
        self.transform = transform
        self.class_to_idx = class_to_idx or {"F": 0, "N": 1}
        self.classes = [label for label, _ in sorted(self.class_to_idx.items(), key=lambda item: item[1])]
        self.samples = []
        self.targets = []
        self.records = []
        missing_files = []

        for row in rows:
            filename = row[REVIEW_CONFIG["filename_column"]]
            label = row[REVIEW_CONFIG["label_column"]]
            image_path = os.path.join(image_dir, filename)
            if not os.path.exists(image_path):
                missing_files.append(filename)
                continue
            target = self.class_to_idx[label]
            self.samples.append((image_path, target))
            self.targets.append(target)
            self.records.append(row)

        if missing_files:
            raise FileNotFoundError(f"metadataにあるのに画像が見つからないファイルがあります: {missing_files[:5]}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, target = self.samples[idx]
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, target

set_reproducibility()
print(f"review_seed:{REVIEW_CONFIG['seed']}")
print(f"deterministic_algorithms:{torch.are_deterministic_algorithms_enabled()}")
print(f"cudnn.deterministic:{torch.backends.cudnn.deterministic}")
print(f"cudnn.benchmark:{torch.backends.cudnn.benchmark}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

review_seed:123
deterministic_algorithms:True
cudnn.deterministic:True
cudnn.benchmark:False


In [ ]:
# モデルの読み込み
best_model_path = os.path.join("train_val", "review_outputs", experiment_name, "models", "model_24.pth")  # best_model_fileに置き換えてください
model = models.efficientnet_b3(weights='IMAGENET1K_V1')
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 2)
model.load_state_dict(torch.load(best_model_path))
model = model.to(device)
model.eval()  # 評価モードに設定

class CustomDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.filenames = os.listdir(data_dir)

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_name = os.path.join(self.data_dir, self.filenames[idx])
        image = Image.open(img_name).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, self.filenames[idx]
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_handles = []

        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        forward_handle = self.target_layer.register_forward_hook(forward_hook)
        backward_handle = self.target_layer.register_backward_hook(backward_hook)

        self.hook_handles.append(forward_handle)
        self.hook_handles.append(backward_handle)

    def generate(self, input_image, target_class=None):
        # 順伝播
        output = self.model(input_image)

        if target_class is None:
            target_class = output.argmax(dim=1).item()

        # 逆伝播
        self.model.zero_grad()
        target = output[0][target_class]
        target.backward()

        # 勾配と活性化を取得
        gradients = self.gradients[0]
        activations = self.activations[0]

        # Grad-CAM
        weights = torch.mean(gradients, dim=[1, 2], keepdim=True)
        grad_cam = torch.sum(weights * activations, dim=0)
        grad_cam = F.relu(grad_cam)
        grad_cam = F.interpolate(grad_cam.unsqueeze(0).unsqueeze(0),
                                 size=input_image.shape[2:], mode="bilinear", align_corners=False)
        grad_cam = grad_cam.squeeze().cpu().numpy()

        # 正規化
        grad_cam = (grad_cam - grad_cam.min()) / (grad_cam.max() - grad_cam.min())
        return grad_cam

    def remove_hooks(self):
        for handle in self.hook_handles:
            handle.remove()

def process_directory_with_grad_cam_using_metadata(
    model,
    directory_path,
    metadata_csv_path,
    split_name,           # "train" or "validation"
    class_names,
    predict_path,
    csv_path
):
    # metadata読み込み
    meta_df = pd.read_csv(metadata_csv_path)

    # 必要なsplitだけ使用
    meta_df = meta_df[meta_df["fixed_family_level_split"] == split_name].copy()

    # label を数値化: F=0, N=1 (Consistent with REVIEW_CONFIG for positive class 'N')
    meta_df["true_label_num"] = meta_df["label"].map({"F": 0, "N": 1})

    # filename -> true label の辞書
    label_map = dict(zip(meta_df["filename"], meta_df["true_label_num"]))
    true_label_name_map = dict(zip(meta_df["filename"], meta_df["label"]))

    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((300, 300)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    os.makedirs(predict_path, exist_ok=True)
    results = []

    target_filenames = meta_df["filename"].tolist()

    for filename in tqdm(target_filenames):
        file_path = os.path.join(directory_path, filename)

        if not os.path.exists(file_path):
            print(f"skip: not found -> {file_path}")
            continue

        image = cv2.imread(file_path)
        if image is None:
            print(f"skip: failed to read -> {file_path}")
            continue

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_tensor = transform(image_rgb).unsqueeze(0).to(device)

        target_layer = model.features[-1]
        grad_cam = GradCAM(model, target_layer)
        cam = grad_cam.generate(input_image=image_tensor, target_class=1) # Specify N (index 1) as target class for grad-cam

        with torch.no_grad():
            outputs = model(image_tensor)
            probabilities = F.softmax(outputs, dim=1)
            predicted_class_idx = torch.argmax(probabilities).item()

            # Probability of 'N' (positive class, index 1) for metrics
            prob_n = probabilities[0, 1].item()
            pred_label = class_names[predicted_class_idx]

        image_rgb_resized = cv2.resize(image_rgb, (300, 300))
        plt.figure(figsize=(12, 4))

        plt.subplot(131)
        plt.imshow(image_rgb_resized)
        plt.title(filename)
        plt.axis("off")

        plt.subplot(132)
        plt.imshow(cam, cmap="jet")
        plt.title("Grad-CAM")
        plt.axis("off")

        plt.subplot(133)
        plt.imshow(image_rgb_resized)
        plt.imshow(cam, cmap="jet", alpha=0.4)
        plt.title(
            f"True: {true_label_name_map[filename]}\n"
            f"Pred: {pred_label}\n"
            f"N_prob={prob_n*100:.2f}%" # Changed to N_prob
        )
        plt.axis("off")

        save_path = os.path.join(predict_path, f"{pred_label}_{filename}.png")
        plt.savefig(save_path)
        plt.close()
        grad_cam.remove_hooks()

        results.append({
            "filename": filename,
            "split": split_name,
            "true_label": true_label_name_map[filename],
            "true_label_num": label_map[filename],
            "predicted_class": pred_label,
            "predicted_class_num": 1 if pred_label == "N" else 0, # N is positive class
            "probability_N": prob_n, # Changed to probability_N
        })

    df = pd.DataFrame(results)
    df.to_csv(csv_path, index=False)
    print(f"✅ 予測結果を保存: {csv_path}")
    accuracy_score = (df["true_label_num"] == df["predicted_class_num"]).mean()

    if len(df) > 0 and df["true_label_num"].nunique() == 2:
        y_true = df["true_label_num"]
        y_pred = df["predicted_class_num"]
        y_score = df["probability_N"] # Probability of class N, which is 1 (positive class)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        # Calculate metrics
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
        f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
        roc_auc = roc_auc_score(y_true, y_score)

        # PR-AUC
        precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_score, pos_label=1)
        pr_auc_score = pr_auc(recall_curve, precision_curve)

        # Bootstrapping for 95% CI of ROC AUC
        n_bootstraps = 1000
        bootstrapped_scores = []

        for i in range(n_bootstraps):
            indices = np.random.choice(len(y_true), len(y_true), replace=True)
            y_true_resample = y_true.iloc[indices]
            y_score_resample = y_score.iloc[indices]

            if y_true_resample.nunique() > 1:
                bootstrapped_scores.append(roc_auc_score(y_true_resample, y_score_resample))
            else:
                pass # Skip if only one class in resample

        if bootstrapped_scores:
            sorted_scores = np.array(bootstrapped_scores)
            sorted_scores.sort()
            ci_lower = sorted_scores[int(0.025 * len(sorted_scores))]
            ci_upper = sorted_scores[int(0.975 * len(sorted_scores))]
        else:
            ci_lower = np.nan
            ci_upper = np.nan

        print(f"accuracy:={accuracy_score:.3f}")
        print(f"📊 Sensitivity: {sensitivity:.3f}")
        print(f"📊 Specificity: {specificity:.3f}")
        print(f"📊 Precision: {precision:.3f}")
        print(f"📊 F1-score: {f1:.3f}")
        print(f"📈 ROC AUC: {roc_auc:.3f} (95% CI: {ci_lower:.3f}-{ci_upper:.3f})")
        print(f"📈 PR-AUC: {pr_auc_score:.3f}")

        metrics = pd.DataFrame([{
            "split": split_name,
            "accuracy": accuracy_score,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Precision": precision,
            "F1-score": f1,
            "AUC": roc_auc,
            "AUC_CI_Lower": ci_lower,
            "AUC_CI_Upper": ci_upper,
            "PR_AUC": pr_auc_score,
        }])
        metrics_path = csv_path.replace(".csv", "_metrics.csv")
        metrics.to_csv(metrics_path, index=False)
        print(f"✅ 評価指標を保存: {metrics_path}")
    else:
        print("⚠️ AUC/confusion matrix/other metrics はクラスが片側しかないため計算しませんでした。")


# testで実行
predict_path = os.path.join(gradcam_root, "test")
csv_path = os.path.join(prediction_csv_path, "gradcam_results_test.csv")
class_names = ["F", "N"]

process_directory_with_grad_cam_using_metadata(
    model=model,
    directory_path=os.path.join("test", "data", "fig"),
    metadata_csv_path=os.path.join("test", "data", "meta","test_metadata.csv"),
    split_name="test",
    class_names=class_names,
    predict_path=predict_path,
    csv_path=csv_path
)

  0%|          | 0/149 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1867: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)
/tmp/ipykernel_21530/545022234.py:75: RuntimeWarning: invalid value encountered in divide
  grad_cam = (grad_cam - grad_cam.min()) / (grad_cam.max() - grad_cam.min())
100%|██████████| 149/149 [01:13<00:00,  2.04it/s]


✅ 予測結果を保存: /content/drive/MyDrive/ScientificReport/re-analysis/version2/test/review_outputs/family_level_split_efficientnetb3_seed123/predictions/tables/gradcam_results_test.csv
accuracy:=0.785
📊 Sensitivity: 0.892
📊 Specificity: 0.474
📊 Precision: 0.832
📊 F1-score: 0.861
📈 ROC AUC: 0.773 (95% CI: 0.678-0.854)
📈 PR-AUC: 0.903
✅ 評価指標を保存: /content/drive/MyDrive/ScientificReport/re-analysis/version2/test/review_outputs/family_level_split_efficientnetb3_seed123/predictions/tables/gradcam_results_test_metrics.csv
